In [1]:
import pandas as pd
import numpy as np
import random

class MarkovTraceGenerator:
    def __init__(self, app_name="SyntheticApp"):
        self.app_name = app_name
        self.transition_matrix = {}
        self.interarrival_distributions = {}
        self.length_distributions = {}
        self.protocols = []

    def set_model(self, transition_matrix, interarrival_dists, length_dists):
        self.transition_matrix = transition_matrix
        self.interarrival_distributions = interarrival_dists
        self.length_distributions = length_dists
        self.protocols = list(transition_matrix.keys())

    def generate(self, n_packets=500, source="172.30.1.1", destination="172.30.1.250"):
        rows = []
        current_proto = random.choice(self.protocols)
        current_time = 0.0

        for i in range(n_packets):
            # Pick length from empirical distribution
            length = int(random.choice(self.length_distributions[current_proto]))

            # Append row
            rows.append({
                "App name": self.app_name,
                "No.": i + 1,
                "Time": round(current_time, 6),
                "Source": source,
                "Destination": destination,
                "Protocol": current_proto,
                "Length": length
            })

            # Sample next protocol
            if current_proto in self.transition_matrix:
                next_proto = random.choices(
                    list(self.transition_matrix[current_proto].keys()),
                    weights=self.transition_matrix[current_proto].values(),
                    k=1
                )[0]
            else:
                next_proto = random.choice(self.protocols)

            # Sample next inter-arrival
            if next_proto in self.interarrival_distributions:
                delta = random.choice(self.interarrival_distributions[next_proto])
            else:
                delta = np.random.exponential(scale=1.0)  # fallback
            current_time += delta

            current_proto = next_proto

        return pd.DataFrame(rows)





In [ ]:
if __name__ == "__main__":
    # ---------- eMBB MODEL ----------
    embb_gen = MarkovTraceGenerator(app_name="eMBB_Synthetic")
    embb_gen.set_model(
        transition_matrix={
            "DNS": {"QUIC": 0.7, "TCP": 0.3},
            "QUIC": {"QUIC": 0.7, "TCP": 0.2, "DNS": 0.1},
            "TCP": {"TCP": 0.6, "QUIC": 0.3, "DNS": 0.1}
        },
        interarrival_dists={
            "DNS": np.random.exponential(scale=5, size=1000).tolist(),   # mean ~5 ms
            "QUIC": np.random.exponential(scale=50, size=1000).tolist(), # mean ~50 ms
            "TCP": np.random.exponential(scale=20, size=1000).tolist()   # mean ~20 ms
        },
        length_dists={
            "DNS": [64, 128],
            "QUIC": [800, 1200, 1500],
            "TCP": [200, 800, 1400]
        }
    )
    embb_trace = embb_gen.generate(n_packets=1000)
    embb_trace.to_csv("embb_markov.csv", index=False)

    # ---------- URLLC MODEL ----------
    urllc_gen = MarkovTraceGenerator(app_name="URLLC_Synthetic")
    urllc_gen.set_model(
        transition_matrix={
            "DNS": {"QUIC": 0.9, "TCP": 0.1},
            "QUIC": {"QUIC": 0.85, "DNS": 0.1, "TCP": 0.05},
            "TCP": {"QUIC": 0.8, "TCP": 0.2}
        },
        interarrival_dists={
            "DNS": np.random.exponential(scale=2, size=1000).tolist(),   # mean ~2 ms
            "QUIC": np.random.exponential(scale=0.5, size=1000).tolist(),# mean ~0.5 ms
            "TCP": np.random.exponential(scale=2, size=1000).tolist()    # mean ~2 ms
        },
        length_dists={
            "DNS": [64, 128],
            "QUIC": [64, 128, 256],
            "TCP": [128, 256, 512]
        }
    )
    urllc_trace = urllc_gen.generate(n_packets=1000)
    urllc_trace.to_csv("urllc_markov.csv", index=False)

    print("✅ Synthetic traces saved as embb_markov.csv and urllc_markov.csv")

✅ Synthetic traces saved as embb_markov.csv and urllc_markov.csv


### Fixed number of rows and fixed timing

In [3]:
import pandas as pd
import numpy as np
import random


class MarkovTraceGenerator:
    def __init__(self, app_name="SyntheticApp"):
        self.app_name = app_name
        self.transition_matrix = {}
        self.interarrival_distributions = {}
        self.length_distributions = {}
        self.protocols = []

    def set_model(self, transition_matrix, interarrival_dists, length_dists):
        """Set transition, interarrival, and length distributions manually"""
        self.transition_matrix = transition_matrix
        self.interarrival_distributions = interarrival_dists
        self.length_distributions = length_dists
        self.protocols = list(transition_matrix.keys())

    # ---------------- Mode 1: Random-timed Markov ----------------
    def generate(self, n_packets=500, source="172.30.1.1", destination="172.30.1.250"):
        """Generate trace with stochastic inter-arrivals"""
        rows = []
        current_proto = random.choice(self.protocols)
        current_time = 0.0

        for i in range(n_packets):
            # Sample packet length
            length = int(random.choice(self.length_distributions.get(current_proto, [0])))

            rows.append({
                "App name": self.app_name,
                "No.": i + 1,
                "Time": round(current_time, 6),
                "Source": source,
                "Destination": destination,
                "Protocol": current_proto,
                "Length": length
            })

            # Transition to next protocol
            if current_proto in self.transition_matrix:
                next_proto = random.choices(
                    list(self.transition_matrix[current_proto].keys()),
                    weights=self.transition_matrix[current_proto].values(),
                    k=1
                )[0]
            else:
                next_proto = random.choice(self.protocols)

            # Inter-arrival time
            if next_proto in self.interarrival_distributions:
                delta = random.choice(self.interarrival_distributions[next_proto])
            else:
                delta = np.random.exponential(scale=1.0)
            current_time += delta

            current_proto = next_proto

        return pd.DataFrame(rows)

    # ---------------- Mode 2: Fixed-timed Markov ----------------
    def generate_fixed(self, n_packets=10000, total_time=10000,
                       source="172.30.1.1", destination="172.30.1.250"):
        """Generate trace with fixed time window and row count"""
        rows = []
        current_proto = random.choice(self.protocols)

        # Equal spacing
        time_step = total_time / (n_packets - 1)

        for i in range(n_packets):
            # Sample packet length (or 0 if no traffic desired)
            length = int(random.choice(self.length_distributions.get(current_proto, [0])))

            rows.append({
                "App name": self.app_name,
                "No.": i + 1,
                "Time": round(i * time_step, 6),
                "Source": source,
                "Destination": destination,
                "Protocol": current_proto,
                "Length": length
            })

            # Transition to next protocol
            if current_proto in self.transition_matrix:
                next_proto = random.choices(
                    list(self.transition_matrix[current_proto].keys()),
                    weights=self.transition_matrix[current_proto].values(),
                    k=1
                )[0]
            else:
                next_proto = random.choice(self.protocols)

            current_proto = next_proto

        return pd.DataFrame(rows)




In [4]:
# ---------------- Example Models ----------------
def make_embb_model():
    gen = MarkovTraceGenerator(app_name="eMBB_Synthetic")
    gen.set_model(
        transition_matrix={
            "DNS": {"QUIC": 0.7, "TCP": 0.3},
            "QUIC": {"QUIC": 0.7, "TCP": 0.2, "DNS": 0.1},
            "TCP": {"TCP": 0.6, "QUIC": 0.3, "DNS": 0.1}
        },
        interarrival_dists={
            "DNS": np.random.exponential(scale=5, size=1000).tolist(),
            "QUIC": np.random.exponential(scale=50, size=1000).tolist(),
            "TCP": np.random.exponential(scale=20, size=1000).tolist()
        },
        length_dists={
            "DNS": [64, 128],
            "QUIC": [800, 1200, 1500],
            "TCP": [200, 800, 1400]
        }
    )
    return gen


def make_urllc_model():
    gen = MarkovTraceGenerator(app_name="URLLC_Synthetic")
    gen.set_model(
        transition_matrix={
            "DNS": {"QUIC": 0.9, "TCP": 0.1},
            "QUIC": {"QUIC": 0.85, "DNS": 0.1, "TCP": 0.05},
            "TCP": {"QUIC": 0.8, "TCP": 0.2}
        },
        interarrival_dists={
            "DNS": np.random.exponential(scale=2, size=1000).tolist(),
            "QUIC": np.random.exponential(scale=0.5, size=1000).tolist(),
            "TCP": np.random.exponential(scale=2, size=1000).tolist()
        },
        length_dists={
            "DNS": [64, 128],
            "QUIC": [64, 128, 256],
            "TCP": [128, 256, 512]
        }
    )
    return gen




In [5]:
if __name__ == "__main__":
    embb_gen = make_embb_model()
    urllc_gen = make_urllc_model()

    # Random-timed traces
    embb_trace_random = embb_gen.generate(n_packets=1000)
    urllc_trace_random = urllc_gen.generate(n_packets=1000)
    embb_trace_random.to_csv("embb_markov_random.csv", index=False)
    urllc_trace_random.to_csv("urllc_markov_random.csv", index=False)

    # Fixed-timed traces (aligned)
    embb_trace_fixed = embb_gen.generate_fixed(n_packets=10000, total_time=10000)
    urllc_trace_fixed = urllc_gen.generate_fixed(n_packets=10000, total_time=10000)
    embb_trace_fixed.to_csv("embb_markov_fixed.csv", index=False)
    urllc_trace_fixed.to_csv("urllc_markov_fixed.csv", index=False)

    print("✅ Generated synthetic traces (random + fixed versions).")


✅ Generated synthetic traces (random + fixed versions).
